In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2007_2008"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "WHQ_E.xpt"
csv_file = DATA_DIR / "WHQ_E_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Keep an untouched copy of what pandas read from XPT.
#
source_df = df.copy()


print("=" * 75)
print("NHANES 2007-2008 WHQ_E XPT -> CSV")
print("=" * 75)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "WHD010",
    "WHD020",
    "WHQ030",
    "WHQ040",
    "WHD045",
    "WHD050",
    "WHQ060",
    "WHQ070",
    "WHD080A",
    "WHD080B",
    "WHD080C",
    "WHD080D",
    "WHD080E",
    "WHD080F",
    "WHD080G",
    "WHD080H",
    "WHD080I",
    "WHD080J",
    "WHD080K",
    "WHD080L",
    "WHD080M",
    "WHD080N",
    "WHD080O",
    "WHD080P",
    "WHD080Q",
    "WHD080R",
    "WHD080S",
    "WHQ270",
    "WHQ280A",
    "WHQ280B",
    "WHQ280C",
    "WHQ280D",
    "WHQ280E",
    "WHQ090",
    "WHD100A",
    "WHD100B",
    "WHD100C",
    "WHD100D",
    "WHD100E",
    "WHD100F",
    "WHD100G",
    "WHD100J",
    "WHD100L",
    "WHD100M",
    "WHD100N",
    "WHD100O",
    "WHD100P",
    "WHD100Q",
    "WHD100R",
    "WHD100S",
    "WHQ210",
    "WHD220",
    "WHD110",
    "WHD120",
    "WHD130",
    "WHD140",
    "WHQ150"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "WHQ_E column names/order differ from expected.\n"
        f"Actual: {df.columns.tolist()}"
    )


if len(df) != 6546:

    raise ValueError(
        f"Unexpected row count: {len(df)}"
    )


if len(df.columns) != 58:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 6,546 rows x 58 columns."
)


# ============================================================
# 4. CHECK FOR KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT zero-decoding artifact.
#
# In this WHQ_E file:
#
#   WHD220 contains 99 occurrences.
#
# CDC defines WHD220 valid range as 0-265.
#
# These tiny values represent numeric zero.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)

true_zero_total = int(
    true_zero_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "True numeric zeros:",
    f"{true_zero_total:,}"
)

print(
    "Tiny artifact occurrences:",
    f"{tiny_total:,}"
)


if tiny_total > 0:

    print("\nTiny values by variable:")

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )


# ============================================================
# 5. VALIDATE TINY VALUES BEFORE CORRECTION
# ============================================================

#
# IMPORTANT:
#
# Do NOT globally replace tiny values.
#
# For this specific WHQ_E file, the only expected
# tiny-value artifact is:
#
# WHD220 = 99 occurrences
#
# If anything else contains the tiny value,
# stop and investigate.
#

expected_tiny_columns = {
    "WHD220": 99
}


actual_tiny_columns = (
    tiny_counts[
        tiny_counts > 0
    ]
    .to_dict()
)


if actual_tiny_columns != expected_tiny_columns:

    raise ValueError(
        "STOP: Tiny-value pattern differs from the "
        "validated WHQ_E pattern.\n"
        f"Expected: {expected_tiny_columns}\n"
        f"Actual: {actual_tiny_columns}"
    )


print(
    "PASS: Tiny-value artifact is isolated "
    "to WHD220 exactly as expected."
)


# ============================================================
# 6. RESTORE WHD220 ZERO VALUES
# ============================================================

#
# CDC WHD220:
#
# Weight loss most successful (pounds)
#
# Valid range = 0 to 265
#
# The 99 tiny values are restored to numeric 0.
#
# This is NOT a blanket tiny-value replacement.
#

wHD220_tiny_mask = (
    df["WHD220"]
    .eq(TINY_VALUE)
)


correction_count = int(
    wHD220_tiny_mask.sum()
)


if correction_count != 99:

    raise ValueError(
        "Unexpected WHD220 tiny-value count."
    )


df.loc[
    wHD220_tiny_mask,
    "WHD220"
] = 0


print("\n--- ZERO RESTORATION ---")

print(
    "WHD220 tiny values restored to zero:",
    correction_count
)


if int(df["WHD220"].eq(0).sum()) != 99:

    raise ValueError(
        "WHD220 zero restoration failed."
    )


print(
    "PASS: 99 WHD220 zero values restored."
)


# ============================================================
# 7. CREATE EXPECTED FIDELITY COPY
# ============================================================

#
# The corrected CSV is expected to be numerically
# identical to the XPT EXCEPT for the validated
# WHD220 tiny-zero restoration.
#

expected_df = source_df.copy()


expected_df.loc[
    expected_df["WHD220"].eq(TINY_VALUE),
    "WHD220"
] = 0


# ============================================================
# 8. CHECK FOR GENUINE DECIMAL VALUES
# ============================================================

#
# CRITICAL FIDELITY RULE:
#
# Never round genuine decimal values.
#
# This check occurs BEFORE conversion to Int64.
#
# If a future file contains legitimate values such as:
#
#     12.5
#     68.7
#     123.45
#
# the program STOPS.
#
# This is especially important for Dietary and
# laboratory files where real decimals must remain
# exactly as supplied.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = df[col].dropna()

    if values.empty:
        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if fractional_mask.any():

        fractional_columns.append(
            col
        )

        print(
            f"\nGenuine decimal values in {col}:"
        )

        print(
            values[
                fractional_mask
            ].head(20)
        )


if fractional_columns:

    raise ValueError(
        "STOP: Genuine decimal values detected. "
        "They will NOT be rounded or changed:\n"
        + str(fractional_columns)
    )


print(
    "PASS: No genuine fractional values "
    "exist in WHQ_E."
)


# ============================================================
# 9. CLEAN SEQN
# ============================================================

seqn = df["SEQN"].dropna()


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")

print(
    "Missing:",
    int(df["SEQN"].isna().sum())
)

print(
    "Duplicates:",
    int(df["SEQN"].duplicated().sum())
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)


if (
    int(df["SEQN"].isna().sum()) != 0
    or
    int(df["SEQN"].duplicated().sum()) != 0
    or
    df["SEQN"].min() != 41475
    or
    df["SEQN"].max() != 51623
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 10. FORMAT WHQ VALUES WITHOUT .0
# ============================================================

#
# WHQ_E contains only whole-number responses in the
# released 2007-2008 file.
#
# The fractional-value test above proves this before
# any conversion is performed.
#
# No .round() is used.
#

DATA_COLUMNS = [
    col
    for col in df.columns
    if col != "SEQN"
]


for col in DATA_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: WHQ_E fields formatted "
    "without unnecessary .0."
)


# ============================================================
# 11. CDC CHECK - MAJOR CATEGORICAL VARIABLES
# ============================================================

#
# Published CDC frequencies.
#

CDC_CATEGORICAL = {

    "WHQ030": {
        "values": {
            1: 3276,
            2: 387,
            3: 2830,
            7: 1,
            9: 10
        },
        "missing": 42
    },

    "WHQ040": {
        "values": {
            1: 557,
            2: 3816,
            3: 2129,
            7: 1,
            9: 1
        },
        "missing": 42
    },

    "WHQ060": {
        "values": {
            1: 724,
            2: 501,
            9: 2
        },
        "missing": 5319
    },

    "WHQ070": {
        "values": {
            1: 1859,
            2: 3918,
            7: 1,
            9: 2
        },
        "missing": 766
    },

    "WHQ270": {
        "values": {
            1: 252,
            2: 2331
        },
        "missing": 3963
    },

    "WHQ090": {
        "values": {
            1: 541,
            2: 3378,
            7: 1,
            9: 1
        },
        "missing": 2625
    },

    "WHQ210": {
        "values": {
            1: 1093,
            2: 2825,
            7: 1,
            9: 2
        },
        "missing": 2625
    }
}


print("\n--- CDC CATEGORICAL CHECKS ---")


for col, expected in CDC_CATEGORICAL.items():

    actual = (
        df[col]
        .value_counts()
        .sort_index()
        .to_dict()
    )

    expected_values = (
        expected["values"]
    )

    expected_missing = (
        expected["missing"]
    )


    if actual != expected_values:

        raise ValueError(
            f"{col} frequencies differ from CDC.\n"
            f"Expected: {expected_values}\n"
            f"Actual: {actual}"
        )


    if int(df[col].isna().sum()) != expected_missing:

        raise ValueError(
            f"{col} missing count differs from CDC."
        )


    print(
        f"PASS: {col} matches CDC."
    )


# ============================================================
# 12. CDC CHECK - WEIGHT/HEIGHT RANGE VARIABLES
# ============================================================

#
# Format:
#
# column:
# (
#   low value,
#   high value,
#   number within normal range,
#   special-code frequencies,
#   missing count
# )
#

CDC_RANGE_CHECKS = {

    "WHD010": (
        53,
        81,
        6363,
        {
            7777: 2,
            9999: 80
        },
        101
    ),

    "WHD020": (
        66,
        470,
        6409,
        {
            7777: 4,
            9999: 47
        },
        86
    ),

    "WHD045": (
        75,
        360,
        4343,
        {
            99999: 18
        },
        2185
    ),

    "WHD050": (
        78,
        470,
        6373,
        {
            7777: 3,
            9999: 88
        },
        82
    ),

    "WHD220": (
        0,
        265,
        3603,
        {
            99999: 37
        },
        2906
    ),

    "WHD110": (
        75,
        505,
        4214,
        {
            7777: 1,
            9999: 159
        },
        2172
    ),

    "WHD120": (
        75,
        450,
        4949,
        {
            7777: 1,
            9999: 231
        },
        1365
    ),

    "WHD130": (
        53,
        78,
        2906,
        {
            7777: 2,
            9999: 118
        },
        3520
    ),

    "WHD140": (
        90,
        551,
        6071,
        {
            7777: 4,
            9999: 88
        },
        383
    ),

    "WHQ150": (
        11,
        79,
        5890,
        {
            80: 128,
            99999: 54
        },
        474
    )
}


print("\n--- CDC RANGE CHECKS ---")


for (
    col,
    (
        low_value,
        high_value,
        expected_range_count,
        special_codes,
        expected_missing
    )
) in CDC_RANGE_CHECKS.items():

    s = df[col]


    actual_range_count = int(
        s.between(
            low_value,
            high_value
        ).sum()
    )


    if actual_range_count != expected_range_count:

        raise ValueError(
            f"{col} range count differs from CDC.\n"
            f"Expected: {expected_range_count}\n"
            f"Actual: {actual_range_count}"
        )


    for code, expected_count in special_codes.items():

        actual_count = int(
            s.eq(code).sum()
        )

        if actual_count != expected_count:

            raise ValueError(
                f"{col} code {code} differs from CDC.\n"
                f"Expected: {expected_count}\n"
                f"Actual: {actual_count}"
            )


    if int(s.isna().sum()) != expected_missing:

        raise ValueError(
            f"{col} missing count differs from CDC."
        )


    print(
        f"PASS: {col} matches CDC."
    )


# ============================================================
# 13. CDC CHECK - WEIGHT LOSS METHODS WHD080
# ============================================================

#
# These variables are "CODE ALL THAT APPLY".
#
# Non-selected responses are stored as missing.
#

CDC_WHD080 = {

    "WHD080A": (10, 1699),
    "WHD080B": (11, 971),
    "WHD080C": (12, 1170),
    "WHD080D": (13, 1519),
    "WHD080E": (14, 462),
    "WHD080F": (15, 273),
    "WHD080G": (16, 104),
    "WHD080H": (17, 124),
    "WHD080I": (31, 69),
    "WHD080J": (32, 149),
    "WHD080K": (33, 20),
    "WHD080L": (40, 47),
    "WHD080M": (34, 1029),
    "WHD080N": (30, 121),
    "WHD080O": (41, 684),
    "WHD080P": (42, 19),
    "WHD080Q": (43, 10),
    "WHD080R": (44, 16),
    "WHD080S": (45, 29)
}


print("\n--- CDC WHD080 CHECKS ---")


for col, (
    expected_code,
    expected_count
) in CDC_WHD080.items():

    s = df[col]


    actual_count = int(
        s.eq(expected_code).sum()
    )


    actual_nonmissing = int(
        s.notna().sum()
    )


    if (
        actual_count != expected_count
        or
        actual_nonmissing != expected_count
    ):

        raise ValueError(
            f"{col} differs from CDC."
        )


    print(
        f"PASS: {col} matches CDC."
    )


# ============================================================
# 14. CDC CHECK - HEALTH PROFESSIONAL TYPES
# ============================================================

CDC_WH280 = {

    "WHQ280A": (1, 81),
    "WHQ280B": (2, 40),
    "WHQ280C": (3, 50),
    "WHQ280D": (4, 85),
    "WHQ280E": (5, 21)
}


print("\n--- CDC WHQ280 CHECKS ---")


for col, (
    expected_code,
    expected_count
) in CDC_WH280.items():

    s = df[col]


    if not (
        int(
            s.eq(
                expected_code
            ).sum()
        )
        ==
        expected_count
        and
        int(
            s.notna().sum()
        )
        ==
        expected_count
    ):

        raise ValueError(
            f"{col} differs from CDC."
        )


    print(
        f"PASS: {col} matches CDC."
    )


# ============================================================
# 15. CDC CHECK - METHODS USED TO AVOID WEIGHT GAIN
# ============================================================

CDC_WHD100 = {

    "WHD100A": (10, 239),
    "WHD100B": (11, 139),
    "WHD100C": (12, 196),
    "WHD100D": (13, 257),
    "WHD100E": (14, 56),
    "WHD100F": (15, 19),
    "WHD100G": (16, 8),
    "WHD100J": (32, 6),
    "WHD100L": (40, 4),
    "WHD100M": (34, 126),
    "WHD100N": (30, 6),
    "WHD100O": (41, 95),
    "WHD100P": (42, 5),
    "WHD100Q": (43, 3),
    "WHD100R": (44, 8),
    "WHD100S": (45, 3)
}


print("\n--- CDC WHD100 CHECKS ---")


for col, (
    expected_code,
    expected_count
) in CDC_WHD100.items():

    s = df[col]


    actual_count = int(
        s.eq(
            expected_code
        ).sum()
    )


    actual_nonmissing = int(
        s.notna().sum()
    )


    if (
        actual_count != expected_count
        or
        actual_nonmissing != expected_count
    ):

        raise ValueError(
            f"{col} differs from CDC."
        )


    print(
        f"PASS: {col} matches CDC."
    )


# ============================================================
# 16. VERIFY WHD220 ZERO RESTORATION
# ============================================================

s = df["WHD220"]


print("\n--- WHD220 ZERO CHECK ---")

print(
    "Zero values:",
    int(s.eq(0).sum())
)

print(
    "Values 0-265:",
    int(s.between(0, 265).sum())
)

print(
    "Don't know (99999):",
    int(s.eq(99999).sum())
)

print(
    "Missing:",
    int(s.isna().sum())
)


if not (
    int(s.eq(0).sum()) == 99
    and
    int(s.between(0, 265).sum()) == 3603
    and
    int(s.eq(77777).sum()) == 0
    and
    int(s.eq(99999).sum()) == 37
    and
    int(s.isna().sum()) == 2906
):

    raise ValueError(
        "WHD220 validation failed."
    )


print(
    "PASS: WHD220 tiny-zero restoration "
    "and CDC frequencies validated."
)


# ============================================================
# 17. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# Expected differences from raw pandas XPT:
#
# WHD220:
#   99 tiny XPORT values -> numeric 0
#
# NOTHING ELSE may change.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    current_values = (
        df[col]
        .astype("Float64")
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        failures.append(
            col
        )


if failures:

    raise ValueError(
        "Unexpected value changes in:\n"
        + str(failures)
    )


print(
    "PASS: All 58 variables preserve expected "
    "XPT values after validated WHD220 zero restoration."
)


# ============================================================
# 18. EXPORT CSV
# ============================================================

#
# IMPORTANT:
#
# No float_format.
# No rounding.
#
# Missing values become blank.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")
print(csv_file)


# ============================================================
# 19. RE-READ EXPORTED CSV
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 20. EXACT CORRECTED XPT -> CSV FIDELITY
# ============================================================

print("\n--- XPT -> CSV FIDELITY ---")


csv_failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "Corrected XPT -> CSV differences found in:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 58 CSV columns match the expected "
    "corrected XPT values."
)


# ============================================================
# 21. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in expected_df.columns:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE CHECK ---")

print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 22. CHECK CSV FOR TINY ARTIFACT
# ============================================================

tiny_csv_count = 0


for col in csv_df.select_dtypes(
    include=[np.number]
).columns:

    tiny_csv_count += int(
        csv_df[col]
        .eq(TINY_VALUE)
        .sum()
    )


print("\n--- CSV TINY VALUE CHECK ---")

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain in CSV."
    )


print(
    "PASS: No tiny-value artifacts remain."
)


# ============================================================
# 23. RAW CSV FORMATTING
# ============================================================

print("\n--- RAW CSV FORMAT CHECK ---")


with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


dot_zero_count = (
    csv_text.count(".0")
)


print(
    "Occurrences of '.0':",
    dot_zero_count
)


if dot_zero_count != 0:

    raise ValueError(
        "Unexpected .0 formatting remains."
    )


print(
    "PASS: No unnecessary .0 formatting."
)


scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print(
    "Scientific notation:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 24. FINAL CSV SEQN CHECK
# ============================================================

print("\n--- FINAL CSV SEQN CHECK ---")


if (
    csv_df["SEQN"].min() != 41475
    or
    csv_df["SEQN"].max() != 51623
    or
    int(csv_df["SEQN"].isna().sum()) != 0
    or
    int(csv_df["SEQN"].duplicated().sum()) != 0
):

    raise ValueError(
        "Final CSV SEQN validation failed."
    )


print(
    "PASS: Final CSV SEQN is valid."
)


# ============================================================
# 25. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 75)
print("WHQ_E CONVERSION COMPLETE")
print("=" * 75)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "Original tiny-value artifacts:",
    tiny_total
)

print(
    "WHD220 zero restorations:",
    correction_count
)

print(
    "Genuine decimal variables: NONE"
)

print(
    "XPT -> CSV unexpected differences: 0 expected"
)

print(
    "Missing-value differences: 0 expected"
)

print(
    "Tiny values remaining: 0 expected"
)

print(
    "Unnecessary .0: 0 expected"
)

NHANES 2007-2008 WHQ_E XPT -> CSV
Rows: 6,546
Columns: 58
PASS: Structure = 6,546 rows x 58 columns.

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifact occurrences: 99

Tiny values by variable:
WHD220    99
PASS: Tiny-value artifact is isolated to WHD220 exactly as expected.

--- ZERO RESTORATION ---
WHD220 tiny values restored to zero: 99
PASS: 99 WHD220 zero values restored.

--- FRACTIONAL VALUE CHECK ---
PASS: No genuine fractional values exist in WHQ_E.

--- SEQN ---
Missing: 0
Duplicates: 0
Minimum: 41475
Maximum: 51623
PASS: SEQN validated.

PASS: WHQ_E fields formatted without unnecessary .0.

--- CDC CATEGORICAL CHECKS ---
PASS: WHQ030 matches CDC.
PASS: WHQ040 matches CDC.
PASS: WHQ060 matches CDC.
PASS: WHQ070 matches CDC.
PASS: WHQ270 matches CDC.
PASS: WHQ090 matches CDC.
PASS: WHQ210 matches CDC.

--- CDC RANGE CHECKS ---
PASS: WHD010 matches CDC.
PASS: WHD020 matches CDC.
PASS: WHD045 matches CDC.
PASS: WHD050 matches CDC.
PASS: WHD220 matches CDC.
PASS